![tower bridge](london.jpg)

London, or as the Romans called it, "Londinium"! Home to [over 8.5 million residents](https://www.ons.gov.uk/peoplepopulationandcommunity/populationandmigration/populationestimates/bulletins/populationandhouseholdestimatesenglandandwales/census2021unroundeddata#population-and-household-estimates-england-and-wales-data) who speak more than [300 languages](https://web.archive.org/web/20080924084621/http://www.cilt.org.uk/faqs/langspoken.htm). While the City of London covers just over one square mile (hence the nickname "The Square Mile"), Greater London has grown to encompass 32 boroughs, spanning a total area of 606 square miles!

![underground train leaving a platform](tube.jpg)

Since the city's streets were originally designed for horses and carriages, this massive growth in both area and population necessitated the development of an efficient public transport system! Since the year 2000, this has been the responsibility of the local government body called **Transport for London**, or *TfL*, overseen by the Mayor of London's office. Its scope includes the Underground (the Tube), Overground trains, the Docklands Light Railway (DLR), buses, trams, river services (clippers and the [Emirates Airline cable car](https://en.wikipedia.org/wiki/London_cable_car)), roads, and even taxis.

The Mayor of London's office makes its data publicly available [here](https://data.london.gov.uk/dataset). In this project, you will work with a slightly modified version of a dataset containing information on passenger journey volumes by transport type.

The data has been loaded into a **Snowflake** database named `TFL`, featuring a single table called `JOURNEYS`, containing the following data:


## TFL.JOURNEYS

| Column | Definition | Data type |
|--------|------------|-----------|
| `MONTH`| Month in numerical format, for example, `1` is January | `INTEGER` |
| `YEAR` | Year | `INTEGER` |
| `DAYS` | Number of days in the specified month | `INTEGER` |
| `REPORT_DATE` | Date on which the data was reported | `DATE` |
| `JOURNEY_TYPE` | Mode of transport used | `VARCHAR` |
| `JOURNEYS_MILLIONS` | Millions of journeys, measured in decimals | `FLOAT` |

In [62]:
-- most_popular_transport_types
    SELECT
        JOURNEY_TYPE,
        ROUND(SUM(JOURNEYS_MILLIONS), 2) AS TOTAL_JOURNEYS_MILLIONS
    FROM TFL.JOURNEYS
    GROUP BY JOURNEY_TYPE
	ORDER BY TOTAL_JOURNEYS_MILLIONS DESC;

,JOURNEY_TYPE,TOTAL_JOURNEYS_MILLIONS
0,Bus,24905.19
1,Underground & DLR,15020.47
2,Overground,1666.85
3,TfL Rail,411.31
4,Tram,314.69
5,Emirates Airline,14.58


In [63]:
import plotly.express as px

# 1. Create a copy of the query result DataFrame
df = most_popular_transport_types.copy()

# 2. Sort ascending so the #1 transport type appears at the top of the chart
df = df.sort_values(by='TOTAL_JOURNEYS_MILLIONS', ascending=True)

# 3. Create the horizontal bar chart
fig = px.bar(
    df,
    x='TOTAL_JOURNEYS_MILLIONS',       
    y='JOURNEY_TYPE',                  
    orientation='h',                  
    text='TOTAL_JOURNEYS_MILLIONS',    
    title='Most Popular Transport Types in London - TfL',
    labels={
        'TOTAL_JOURNEYS_MILLIONS': 'Total Journeys (Millions)',
        'JOURNEY_TYPE': 'Transport Type'
    },
    color='TOTAL_JOURNEYS_MILLIONS',   
    color_continuous_scale='Teal'      
)

# 4. Format numbers with thousands separator and 'M' suffix outside each bar
fig.update_traces(
    texttemplate='%{text:,.2f} M',    
    textposition='outside'            
)

# 5. Layout configuration with dark theme and dynamic X-axis padding
max_value = df['TOTAL_JOURNEYS_MILLIONS'].max()
fig.update_layout(
    template='plotly_dark',
    xaxis_title='Total Journeys (Millions of Passengers)',
    yaxis_title='Transport Type',
    coloraxis_showscale=False,        
    xaxis=dict(range=[0, max_value * 1.18]),  
    margin=dict(l=20, r=40, t=60, b=40)
)

fig.show()

In [64]:
-- emirates_airline_popularity
SELECT 
   MONTH, 
   YEAR, 
   ROUND(JOURNEYS_MILLIONS, 2) AS ROUNDED_JOURNEYS_MILLIONS
FROM TFL.JOURNEYS
WHERE JOURNEY_TYPE = 'Emirates Airline' 
AND JOURNEYS_MILLIONS IS NOT NULL
ORDER BY ROUNDED_JOURNEYS_MILLIONS DESC
LIMIT 5;

,MONTH,YEAR,ROUNDED_JOURNEYS_MILLIONS
0,5,2012,0.53
1,6,2012,0.38
2,4,2012,0.24
3,5,2013,0.19
4,5,2015,0.19


In [65]:
# 1. Create a copy of the query result DataFrame
df = emirates_airline_popularity.copy()

# 2.  Format columns  year / month 
df['MONTH_YEAR'] = df['MONTH'].astype(str).str.zfill(2) + '/' + df['YEAR'].astype(str)

# 3. Order by the largerst value 
df = df.sort_values(by='ROUNDED_JOURNEYS_MILLIONS', ascending=True)

# 4. Create the horizontal bar chart
fig = px.bar(
    df,
    x='ROUNDED_JOURNEYS_MILLIONS',       
    y='MONTH_YEAR',                      
    orientation='h',                     
    text='ROUNDED_JOURNEYS_MILLIONS',    
    title='Top 5 Most Popular Months and Years - Emirates Airline',
    labels={
        'ROUNDED_JOURNEYS_MILLIONS': 'Journeys (Millions of Passengers)',
        'MONTH_YEAR': 'Month / Year'
    },
    color='ROUNDED_JOURNEYS_MILLIONS',   
    color_continuous_scale='Viridis'    
)

# 5.  Format numbers with thousands separator and 'M' suffix outside each bar
fig.update_traces(
    texttemplate='%{text:.2f} M',        
    textposition='outside'              
)

# 6. Layout configuration with dark theme and dynamic X-axis padding
fig.update_layout(
    template='plotly_dark',
    xaxis_title='Journeys (Millions of Passengers)',
    yaxis_title='Month / Year',
    coloraxis_showscale=False,          
    xaxis=dict(range=[0, 0.62])         
)

fig.show()


In [66]:
-- least_popular_years_tube
SELECT 
    YEAR, 
    JOURNEY_TYPE, 
    SUM(JOURNEYS_MILLIONS) AS TOTAL_JOURNEYS_MILLIONS 
FROM TFL.JOURNEYS 
WHERE JOURNEY_TYPE = 'Underground & DLR' 
GROUP BY YEAR, JOURNEY_TYPE 
ORDER BY TOTAL_JOURNEYS_MILLIONS ASC 
LIMIT 5;


,YEAR,JOURNEY_TYPE,TOTAL_JOURNEYS_MILLIONS
0,2020,Underground & DLR,310.179316
1,2021,Underground & DLR,748.452544
2,2022,Underground & DLR,1064.859009
3,2010,Underground & DLR,1096.145588
4,2011,Underground & DLR,1156.647654


In [67]:
# 1. Create a copy of the query result DataFrame
df = least_popular_years_tube.copy()

# 2. Convert YEAR to string so Plotly treats it as discrete categories, not continuous numbers
df['YEAR'] = df['YEAR'].astype(str)

# 3. Sort ascending so the year with the absolute lowest volume appears at the top
df = df.sort_values(by='TOTAL_JOURNEYS_MILLIONS', ascending=True)

# 4. Create the horizontal bar chart
fig = px.bar(
    df,
    x='TOTAL_JOURNEYS_MILLIONS',        
    y='YEAR',                         
    orientation='h',                  
    text='TOTAL_JOURNEYS_MILLIONS',   
    title='Least Popular Years - Underground & DLR',
    labels={
        'TOTAL_JOURNEYS_MILLIONS': 'Total Journeys (Millions)',
        'YEAR': 'Year'
    },
    color='TOTAL_JOURNEYS_MILLIONS',   
    color_continuous_scale='Teal'      
)

# 5. Format labels outside the bars with thousands separator and 'M' suffix
fig.update_traces(
    texttemplate='%{text:,.2f} M',     
    textposition='outside'            
)

# 6. Apply dark theme styling and dynamic axis margin to avoid label truncation
max_value = df['TOTAL_JOURNEYS_MILLIONS'].max()
fig.update_layout(
    template='plotly_dark',
    xaxis_title='Total Journeys (Millions of Passengers)',
    yaxis_title='Year',
    coloraxis_showscale=False,       
    xaxis=dict(range=[0, max_value * 1.18]), 
    yaxis=dict(autorange="reversed"), 
    margin=dict(l=20, r=40, t=60, b=40)
)

fig.show()
